# Dynamic risk update

Diagnosztikai evidencia → posterior LT-101 fault probability → fault-tree risk branch.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from safetycourse.diagnosis import residual, ewma
from safetycourse.bayes import BinaryNode, BinaryBayesNet

In [ ]:
df=pd.read_csv('../data/pundit_case/sensor_data.csv')
df['residual']=residual(df.lt101_measured_pct,df.true_level_pct)
df['ewma']=ewma(df.residual,0.12)
df['mismatch']=df.ewma.abs()>4
net=BinaryBayesNet([BinaryNode('F',[],{():0.02}),BinaryNode('E',['F'],{(False,):0.05,(True,):0.90})])
df['p_fault']=[net.query('F',{'E':bool(e)})[True] for e in df.mismatch]

In [ ]:
p_challenge=0.08
p_auto=1-(1-0.01)*(1-0.005)
p_operator=0.04
def p_overflow(pf):
    p_manual=1-(1-pf)*(1-p_operator)
    return p_challenge*p_auto*p_manual
df['p_overflow']=df.p_fault.map(p_overflow)
df[['time_min','ewma','p_fault','p_overflow']].tail()

In [ ]:
plt.plot(df.time_min,df.p_overflow)
plt.xlabel('time [min]'); plt.ylabel('P(overflow | evidence)'); plt.show()

Ez minimum demonstráció: nem continuous Bayesian filter, és a likelihood értékek kalibrálandó paraméterek.